# Instalar dependencias

In [4]:
!pip install fitz pytesseract openai PyMuPDF pdfplumber

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 70.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 81.5 MB/s eta 0:00:00


# Imports

In [5]:
# ===== PDF processing =====
import fitz  # PyMuPDF
import pdfplumber

# ===== OCR =====
import pytesseract
from PIL import Image
import cv2
import numpy as np

# ===== Data handling =====
import pandas as pd
import json
from pathlib import Path

# ===== LLM interaction =====
from openai import OpenAI

# ===== Utilities =====
import re
import os
from typing import List, Dict

## load_pdfs()
Función para cargar cargar archivos PDF. Toma una ruta de folder para retornar todos los PDFs dentro en una lista.

In [6]:
def load_pdfs(folder_path: str) -> List[Path]:
    """
    Returns a list of PDF file paths from a folder.
    """
    folder = Path(folder_path)
    pdf_files = list(folder.glob("*.pdf"))
    return pdf_files

In [7]:
def pdf_has_text(pdf_path, max_pages=5, threshold=20):
    """
    Checks if a PDF contains embedded text.

    Parameters
    ----------
    pdf_path : str or Path
        Path to the PDF file.
    max_pages : int
        Number of pages to check (for speed).

    Returns
    -------
    bool
        True if text is detected, False otherwise.
    """
    import fitz

    with fitz.open(pdf_path) as doc:
        pages_to_check = min(max_pages, len(doc))

        for page_num in range(pages_to_check):
            page = doc.load_page(page_num)
            text = page.get_text().strip()

            if len(text) > threshold:  # threshold to avoid false positives
                return True

    return False

In [8]:
def extract_text_pdf(pdf_path):
    """
    Extracts text from a PDF that already contains embedded text.
    """
    import fitz

    text = ""

    with fitz.open(pdf_path) as doc:
        for page in doc:
            text += page.get_text()

    return text

In [9]:
def extract_text_ocr(pdf_path):
    """
    Extracts text from a scanned PDF using OCR.
    """

    text = ""

    with fitz.open(pdf_path) as doc:
        for page in doc:

            # render page as image
            pix = page.get_pixmap(dpi=300)

            img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
            img = np.array(img)

            # convert to grayscale (helps OCR)
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

            page_text = pytesseract.image_to_string(gray)

            text += page_text + "\n"

    return text

In [10]:
results = []
path = "some/path"

for doc in load_pdfs(path):

  if pdf_has_text(doc):
    text = extract_text_pdf(doc)

  else:
    text = extract_text_ocr(doc)


  results.append({
        "file_name": doc["file_name"],
        "text": text
    })

df = pd.DataFrame(results)
df.head()

""


In [ ]:
def extract_with_llm(df, client, model="gpt-4.1-mini", max_chars=12000):
    """
    Sends extracted text to an LLM and returns structured data.
    """

    outputs = []

    for _, row in df.iterrows():

        file_name = row["file_name"]
        text = row["text"][:max_chars]  # truncate for safety

        prompt = f"""
        Extract the following information from this document.

        Return ONLY valid JSON.

        Fields:
        - document_type
        - organization
        - date
        - total_amount
        - summary

        Document:
        {text}
        """

        try:

            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": "You extract structured data from documents."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0
            )

            result = response.choices[0].message.content

        except Exception as e:

            result = f"ERROR: {str(e)}"

        outputs.append({
            "file_name": file_name,
            "llm_output": result
        })

    return pd.DataFrame(outputs)

In [ ]:
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [ ]:
llm_results = extract_with_llm(df, client)

In [ ]:
llm_results.head()

In [ ]:
json.loads(result)